# GBM WSI QC calibration

Open this notebook from the repository root using the patching environment. It reads existing
tiling output and previews the policy on TCGA training patients for one validation fold. It
does not re-tile slides or show MGMT labels. Run one fold at a time, with the shared fold map.

Change the policy YAML to test other QC settings. Review a seeded sample of accepted, review,
and rejected patches, then look around the candidate blur boundaries. In pale and low-texture
patches, check for readable nuclei and tissue architecture at the original tile scale.

In [ ]:
from pathlib import Path
import csv
import random
import sys

import matplotlib.pyplot as plt
import pandas as pd

REPO = Path.cwd()
ROOT = Path("/workspace/persist/crivas4-microscopy/GBM_MGMT")
SUBSET = "A"           # A, B or full
VALIDATION_FOLD = 0    # TCGA fold to hold out
POLICY = REPO / "configs/qc_policy.yaml"
TILING = REPO / "configs/tiling.yaml"
FOLD_MAP = REPO / "manifests/patient_folds.csv"

sys.path.insert(0, str(REPO / "data_prep"))
import common
import loader
import qc
import review

policy, policy_hash, _ = qc.load_policy(POLICY)
if policy["validation_fold"] not in (None, VALIDATION_FOLD):
    raise ValueError("The policy belongs to another validation fold.")

fold_hash = common.sha256_file(FOLD_MAP)
folds = common.read_folds(FOLD_MAP)
if common.sha256_file(FOLD_MAP) != fold_hash:
    raise RuntimeError("Fold map changed while reading.")
training_ids = qc.calibration_patients(folds, VALIDATION_FOLD)
qc.validate_calibration(policy, folds)
tiling_version = common.tiling_version(TILING)
index_path = ROOT / "patches" / tiling_version / f"index_{SUBSET}.csv"
training_slides = [
    row for row in common.read_rows(index_path)
    if row["source"] == "TCGA" and row["patient_id"] in training_ids
]
patients = sorted({row["patient_id"] for row in training_slides})
if not patients:
    raise ValueError(f"No eligible training patients in {index_path}")

journal = (ROOT / "review" / tiling_version / f"calibration_fold_{VALIDATION_FOLD}"
           / f"{policy_hash}_{fold_hash}.csv")
journal_fields = [
    "patient_id", "slide_id", "patch_id", "decision", "automatic_reasons",
    "usable", "confirmed_defect", "reviewer", "notes"
]
print(f"Fold {VALIDATION_FOLD}: {len(patients)} training patients, {len(training_slides)} slides")
print(patients)

## Inspect patches

Pick a training patient, then set a decision, an automatic reason, a confirmed defect,
or a blur band. Leave a filter as `None` to include all values. Use the same seed to
repeat a sample, and change the seed for another. The same score can represent technical
blur or sharp but smooth tissue. A score near a proposed cutoff is a reason to inspect the image.

Cached patches are checked against their original pixel hashes. Uncached patches are
reconstructed from the WSI and checked against the same hash. A missing WSI or unreadable
patch is reported on its panel.

In [ ]:
PATIENT = patients[0]             # or paste an ID from the list above
DECISION = None                   # accept, review, reject, or None
AUTOMATIC_REASON = None           # e.g. low_blur, blur, pale, invalid_score
CONFIRMED_DEFECT = None           # e.g. blur, fold, pen, or None
BLUR_BAND = None                  # e.g. (50, 100), lower inclusive, upper exclusive
N = 12
SEED = 42

if PATIENT not in patients:
    raise ValueError("Choose a training patient in this subset.")
if N < 1:
    raise ValueError("N must be positive.")

previous = {row["patch_id"]: row for row in common.read_rows(journal)} if journal.exists() else {}
patient_slides = [s for s in training_slides if s["patient_id"] == PATIENT]
rng = random.Random(SEED)
sample = []
matches = 0

for slide in patient_slides:
    with (ROOT / slide["csv"]).open(newline="") as handle:
        for patch in csv.DictReader(handle):
            if patch["role"] not in qc.ROLES:
                continue
            decision, reasons = qc.decide(patch, policy)
            tags = [reason.split(":", 1)[0] for reason in reasons.split(";")]
            confirmed = previous.get(patch["patch_id"], {}).get("confirmed_defect", "")
            if DECISION is not None and decision != DECISION:
                continue
            if AUTOMATIC_REASON is not None and AUTOMATIC_REASON not in tags:
                continue
            if CONFIRMED_DEFECT is not None and confirmed != CONFIRMED_DEFECT:
                continue
            if BLUR_BAND is not None:
                scores, _ = qc.read_scores(patch)
                focus = scores.get("laplacian_variance")
                if focus is None or not BLUR_BAND[0] <= focus < BLUR_BAND[1]:
                    continue
            matches += 1
            slot = rng.randrange(matches) if len(sample) == N else len(sample)
            if slot < N:
                item = dict(patch, h5=slide["h5"], decision=decision, automatic_reasons=reasons,
                            confirmed_defect=confirmed)
                if slot == len(sample):
                    sample.append(item)
                else:
                    sample[slot] = item

print(f"{matches:,} matches, {len(sample)} sampled")
if sample:
    fig, axes = plt.subplots((len(sample) + 3) // 4, 4, figsize=(14, 3.5 * ((len(sample) + 3) // 4)))
    for ax in axes.flat:
        ax.axis("off")
    with loader.TileReader(ROOT, tiling_config=TILING) as reader:
        for number, (ax, patch) in enumerate(zip(axes.flat, sample)):
            try:
                ax.imshow(reader.read(patch))
            except (RuntimeError, OSError, ValueError, KeyError) as exc:
                ax.text(0.5, 0.5, "Read failed", ha="center")
                print(patch["patch_id"], exc)
            ax.set_title(f"{number}  {patch['decision']}  |  focus {patch.get('laplacian_variance', 'missing')}\n"
                         f"{patch['automatic_reasons']}  |  {patch['confirmed_defect'] or 'unconfirmed'}",
                         fontsize=9, wrap=True)
    fig.tight_layout()
    plt.show()
    display(pd.DataFrame(sample).reindex(columns=[
        "patch_id", "slide_id", "x0", "y0", "decision", "automatic_reasons",
        "laplacian_variance", "confirmed_defect", "stored"
    ]))
else:
    print("No matches. Try clearing a filter or choosing another training patient.")

## Record a calibration observation

Choose a row number from the montage. Record whether the patch is usable, unusable, or
uncertain, and enter a defect only if you confirmed it. Use `none` when no defect is confirmed.
This journal is scoped to the fold, fold map and policy preview. It is a calibration record.
The production review and sign-off workflow in `data_prep/review.py` remains available
for artifact masks, slide sign-off and final patch decisions.

In [ ]:
SAVE = False
ROW_NUMBER = 0
USABLE = "uncertain"             # usable, unusable or uncertain
CONFIRMED = "none"               # allowed values below
REVIEWER = ""
NOTES = ""

print("Confirmed defect choices:", ", ".join(sorted(review.DEFECTS)))
if SAVE:
    if not 0 <= ROW_NUMBER < len(sample):
        raise ValueError("Choose an image shown in the current montage.")
    if not REVIEWER.strip() or USABLE not in {"usable", "unusable", "uncertain"} or CONFIRMED not in review.DEFECTS:
        raise ValueError("Enter a reviewer, usability decision and valid defect type.")
    patch = sample[ROW_NUMBER]
    review._append(journal, journal_fields, {
        "patient_id": patch["patient_id"], "slide_id": patch["slide_id"], "patch_id": patch["patch_id"],
        "decision": patch["decision"], "automatic_reasons": patch["automatic_reasons"],
        "usable": USABLE, "confirmed_defect": CONFIRMED, "reviewer": REVIEWER.strip(), "notes": NOTES,
    })
    print("Saved", patch["patch_id"])
if journal.exists():
    observations = pd.read_csv(journal).drop_duplicates("patch_id", keep="last")
    print("Reviewed patient IDs:", sorted(observations["patient_id"].unique()))
    display(observations.groupby(["usable", "confirmed_defect"]).size().rename("patches"))

## Compare proposed blur cutoffs

Each pair is `(reject below, review below)`. Other rules stay fixed. Inspect the accepted,
review and rejected counts per slide and patient. `accepted_cached` shows how many of
the accepted tiles have pixels already in the cache. These are automatic QC counts
before any human override. A sweep over the whole training subset reads all its tissue scores.

Choose cutoffs after examining reviewer decisions and per-patient retention, including
pale and low-texture tissue. Set `validation_fold`, `calibration_patients`, `reviewer` and
`notes` in a new policy YAML. Record all training patients whose slides or retention
results informed the choice. Freeze the policy after the team approves the evidence.
Use that same policy for A and B within the fold.

In [ ]:
from collections import Counter

SWEEP_PATIENTS = [PATIENT]         # use patients for the whole indexed training subset
CANDIDATES = [(50, 100), (75, 100), (100, 100)]
if not set(SWEEP_PATIENTS) <= set(patients):
    raise ValueError("Sweep patients must be training patients in this subset.")
if any(not 0 <= reject <= check for reject, check in CANDIDATES):
    raise ValueError("Each pair must satisfy 0 <= reject cutoff <= review cutoff.")
trial_policies = [
    dict(policy, blur_reject_below=reject, blur_review_below=check)
    for reject, check in CANDIDATES
]
results = []
for slide in training_slides:
    if slide["patient_id"] not in SWEEP_PATIENTS:
        continue
    counts = [Counter() for _ in CANDIDATES]
    with (ROOT / slide["csv"]).open(newline="") as handle:
        for patch in csv.DictReader(handle):
            if patch["role"] != "tissue":
                continue
            for trial, tally in zip(trial_policies, counts):
                decision, _ = qc.decide(patch, trial)
                tally[decision] += 1
                if decision == "accept" and patch["stored"] == "True":
                    tally["accepted_cached"] += 1
    for (reject, check), tally in zip(CANDIDATES, counts):
        results.append(dict(patient_id=slide["patient_id"], slide_id=slide["slide_id"],
                            reject_below=reject, review_below=check, accept=tally["accept"],
                            review=tally["review"], reject=tally["reject"],
                            accepted_cached=tally["accepted_cached"]))

if results:
    slide_counts = pd.DataFrame(results)
    keys = ["patient_id", "reject_below", "review_below"]
    columns = ["accept", "review", "reject", "accepted_cached"]
    patient_counts = slide_counts.groupby(keys, as_index=False)[columns].sum()
    patient_counts["accepted_fraction"] = patient_counts["accept"].div(
        patient_counts[["accept", "review", "reject"]].sum(axis=1)
    )
    display(slide_counts)
    display(patient_counts)
else:
    print("No slides selected.")